In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colorbar as colorbar
import matplotlib.colors as mcolors
import matplotlib.patches as patches
from itertools import product
import seaborn as sns
import torch.utils.benchmark as benchmark
from tqdm.notebook import tqdm, trange
import pandas as pd
import os
from scipy.optimize import curve_fit

import plotly.io as pio
pio.renderers.default = "notebook_connected"
import plotly.graph_objects as go

# from run_exp import run_exp
# import linear_algebra_utils as lau
from icl.linear import plot_task_vector_differences, plot_task_vector_variance_with_fit, plot_pairwise_task_vector_variance
from icl.utils.train import BaseTrainer, train_model_with_plot
from icl.config import get_config_base
from icl.tasks import LatentMarkov, LatentIDBayes, LatentOODBayes
from icl.models import Transformer
from icl.utils import visualize_attention
from icl.utils.train_utils import get_attn_base, compute_cross_entropy
from icl.figures.head_view import *
import icl.utils.task_vec as task_vec
import icl.utils.notebook_utils as nu

torch.set_printoptions(precision=3, sci_mode=False)

%load_ext autoreload
%autoreload 2

In [2]:
config = get_config_base()
model = Transformer(config)
model = model.to(config.device)
_ = train_model_with_plot(model, config, show=False)

Experiment directory:  ../results/reversion/train_44a705a6aa210fcbb4f84bf3c101b556


wandb: Currently logged in as: hyan84 (hyan84-university-of-wisconsin-madison) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Training: 236.852345 s
Loss plots saved at ../results/reversion/train_44a705a6aa210fcbb4f84bf3c101b556/plots
../results/reversion/train_44a705a6aa210fcbb4f84bf3c101b556/plots/loss.png


In [5]:
for k in trange(500, 1000, 100):
    config = get_config_base()
    config.task.total_trans = k
    model = Transformer(config)
    model = model.to(config.device)
    
    _ = train_model_with_plot(model, config, show=False)

  0%|          | 0/5 [00:00<?, ?it/s]

Experiment directory:  results/reversion/train_70ca28e67abd41ec38d7c2d1d2b89009
train_70ca28e67abd41ec38d7c2d1d2b89009 already completed
Experiment directory:  results/reversion/train_9d9be38dfdf8f1063551caca3bd82188
train_9d9be38dfdf8f1063551caca3bd82188 already completed
Experiment directory:  results/reversion/train_990ed1f600654c4e6bab1a36bc11dd7d
train_990ed1f600654c4e6bab1a36bc11dd7d already completed
Experiment directory:  results/reversion/train_847dcf6819283bf8cf7acd122998be23
train_847dcf6819283bf8cf7acd122998be23 already completed
Experiment directory:  results/reversion/train_bb470aacf85210a583b2f1551b2cb8e6
train_bb470aacf85210a583b2f1551b2cb8e6 already completed


In [18]:
model, sampler, config = nu.load_everything("reversion", "train_9814d096e594f3956b478f42debca1ac")

In [22]:
sample, _ = sampler.generate(mode="test", num_samples=1)
attns = get_attn_base(model, sample)
cap = 140
attns_capped = {layer_key: tensor.squeeze(0)[:, :cap, :cap] for layer_key, tensor in attns.items()}
widget = visualize_attention(attns_capped, mode='widget', residual_mode='without')
widget

In [21]:
batch, info = sampler.generate(mode="test", num_samples=1)
nu.view_mask(batch, info)

In [32]:
from icl.figures.head_view import get_head_view
get_head_view(model, config, sampler=sampler, trunc=90, action="view")

<IPython.core.display.Javascript object>